# 13 Exporting with Ease

## 13.1 Figures

In [ ]:
import csv
import pickle as pkl
from pathlib import Path
from glob import glob

# from osgeo import gdal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from netCDF4 import Dataset

In [ ]:
# from PIL import Image
import numpy as np

In [ ]:
plt.rcParams.update({"font.size": 16, "figure.figsize": (6, 6), "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]})

In [ ]:
filename = "hms_fire20250109.txt"
fires_path = Path("./data/txt") / filename
fires = pd.read_csv(fires_path, skipinitialspace=True)

In [ ]:
bins_10 = np.arange(0, 1000, 10)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.hist(fires["FRP"], bins=bins_10)
ax.set_yscale("log")
ax.set_xlabel("Fire Radiative Power (MW)")
ax.set_ylabel("Counts")

plt.savefig("hms_fire20250109_histogram.png",  \
    bbox_inches="tight", pad_inches=0.5, \
    dpi=300, facecolor='yellow')
plt.close()

## 13.2 Exporting Data

In [ ]:
name = ["GOES-19", "IceSat-2", "Himawari-9"]
agency = ["NOAA", "NASA", "JAXA"]
orbit = ["GEO", "LEO", "GEO"]

In [ ]:
with open("satellite.txt","w", newline="") as csvfile:
    writer = csv.writer(csvfile, delimiter=",")
    
    # adds a header
    writer.writerow(["name", "agency", "orbit"])
    
    # Writes each element as a single row
    for i in range(0, len(name)):
        writer.writerow([name[i], agency[i], orbit[i]])

In [ ]:
satellites = pd.DataFrame({
    "name": name,
    "agency": agency,
    "orbit": orbit})

satellites.to_csv("satellites.csv",index=False)

## 13.3 Pickling

In [ ]:
pkl.dump(satellites, open("satellites.p", "wb"))

In [ ]:
satellites2 = pkl.load(open("satellites.p", "rb"))
satellites2

## 13.4 NumPy binary files

In [ ]:
np.savez("satnames", name=name, agency=agency, orbit=orbit)

In [ ]:
npzfile = np.load("satnames.npz")
npzfile.files

In [ ]:
npzfile["name"]

In [ ]:
npzfile.close()

## 13.5 NetCDF

### 13.5.1. Using netCDF4 to Create netCDF Files

In [ ]:
aod_path = Path("./data/aod")
filenames = list(aod_path.glob("JRR-AOD_v3r2_j01_s2023060718*.nc"))

file_id = Dataset(filenames[0], mode="r")

In [ ]:
for var in ["Latitude", "Longitude", "AOD550", "QCAll"]:
    print(var, file_id.variables[var].dimensions)

In [ ]:
for dim in ["Columns", "Rows"]:
    print(dim, len(file_id.dimensions[dim]))

In [ ]:
cols = len(file_id.dimensions["Columns"])
rows = len(file_id.dimensions["Rows"])
num_files = len(filenames)
file_id.close()

print(rows, cols, num_files)

In [ ]:
output_filename = Path("JRR-AOD-combined.nc")

rootgrp = Dataset(output_filename, "w", \
    format="NETCDF4")
rootgrp.description = "Combined NOAA-20 AOD swaths"

In [ ]:
lat = rootgrp.createDimension("lat", rows*num_files)
lon = rootgrp.createDimension("lon", cols)

In [ ]:
latitudes = rootgrp.createVariable( \
    "Latitude","f4", ("lat", "lon"), \
    zlib = True, least_significant_digit = 2)

longitudes = rootgrp.createVariable( \
    "Longitude","f4", ("lat", "lon"), \
    zlib = True, least_significant_digit = 2)

variable = rootgrp.createVariable( \
    "AOD550", "f4", ("lat", "lon"), \
    zlib = True, least_significant_digit = 2, \
    fill_value = -999.9)

In [ ]:
latitudes.units = "degrees north"
longitudes.units = "degrees east"

In [ ]:
var = np.zeros((rows*num_files, cols))
lats = np.zeros((rows*num_files, cols))
lons = np.zeros((rows*num_files, cols))

for filenum, filename in enumerate(filenames):
    file_id = Dataset(filename, mode="r")
    
    i1 = rows*filenum
    i2 = rows*(filenum+1)
    
    # Get coordinates
    lats[i1:i2, 0:cols] = file_id.variables["Latitude"][:,:]
    lons[i1:i2, 0:cols] = file_id.variables["Longitude"][:,:]
        
    # Create and fill variables
    value = file_id.variables["AOD550"][:,:]

    # Simple binary quality flag    
    dqf = file_id.variables[ "QCAll"][:,:]
    value[ dqf != 0 ] = -999.9
    
    var[i1:i2, :] = value[:,:]

    file_id.close()

In [ ]:
latitudes[:,:] = lats
longitudes[:,:] = lons
variable[:,:] = var

In [ ]:
rootgrp.close()

### 13.5.2. Using Xarray to Create netCDF Files

In [ ]:
aod = xr.open_mfdataset(filenames, concat_dim="Rows", \
    combine="nested", data_vars='all')
filtered = aod["AOD550"].where(aod["QCAll"] == 0)

In [ ]:
encoding = {
    "AOD550": {"zlib": True, "least_significant_digit": 2},
    "Latitude": {"zlib": True, "least_significant_digit": 2},
    "Longitude": {"zlib": True, "least_significant_digit": 2}
}

filtered.to_netcdf("JRR-AOD-xarray-combined.nc", encoding=encoding)

In [ ]:
del(filtered.attrs["units"])